# 312 — Graph Tools

Tests every MCP tool against live Neo4j data.

Uses real entity IDs confirmed from the CSV data:
- `LOAN-0002` — LVR=95, high-risk loan
- `BRW-0001` — Patrick Nelson, individual, high risk
- `ACC-0596` — transaction structuring target
- `BRW-0582` — top of layered ownership chain

In [ ]:
%run 311_agent_setup.ipynb

## Tool 1: read-neo4j-cypher (Neo4j MCP)

In [ ]:
import json

result = execute_tool('read-neo4j-cypher', {
    'query': """
        MATCH (l:LoanApplication {loan_id: 'LOAN-0002'})-[:SUBMITTED_BY]->(b:Borrower)
        RETURN l.loan_id AS loan_id, l.lvr AS lvr, l.amount AS amount,
               b.borrower_id AS borrower_id, b.name AS name, b.risk_rating AS risk
    """
})
print('read-neo4j-cypher:', json.dumps(result, indent=2, default=str))

## Tool 2: traverse_compliance_path (FastMCP)

In [ ]:
result = execute_tool('traverse_compliance_path', {
    'entity_id': 'LOAN-0002',
    'entity_type': 'LoanApplication',
    'regulation_id': 'APG-223',
})

print('Entity:', result.get('entity', {}))
print('Jurisdiction:', result.get('jurisdiction_id'))
print('\nRegulations found:', list(result.get('regulations', {}).keys()))

for reg_id, reg in result.get('regulations', {}).items():
    for sec_id, sec in reg.get('sections', {}).items():
        for req_id, req in sec.get('requirements', {}).items():
            if req.get('thresholds'):
                for t in req['thresholds']:
                    print(f'  {t["threshold_id"]}: {t["metric"]} {t["operator"]} {t["value"]} {t["unit"]}')

## Tool 3: retrieve_regulatory_chunks (FastMCP — vector search)

In [ ]:
result = execute_tool('retrieve_regulatory_chunks', {
    'query_text': 'LVR limit high risk residential mortgage lending',
    'regulation_id': 'APG-223',
    'top_k': 3,
})

print(f'Query: {result["query"]}')
for chunk in result.get('chunks', []):
    print(f'\n[{chunk["chunk_id"]}] score={chunk["similarity_score"]}')
    print(f'  {chunk["text"][:200]}...')

## Tool 4: detect_graph_anomalies (FastMCP) — all 6 patterns

In [ ]:
patterns = [
    'transaction_structuring',
    'high_lvr_loans',
    'high_risk_industry',
    'layered_ownership',
    'high_risk_jurisdiction',
    'guarantor_concentration',
]

for pattern in patterns:
    result = execute_tool('detect_graph_anomalies', {'pattern_name': pattern})
    count = result.get('finding_count', 0)
    sev   = result.get('severity', '?')
    ids   = result.get('entity_ids', [])[:5]
    status = '✓' if count > 0 else '○'
    print(f'{status} [{sev}] {pattern}: {count} findings  {ids}')

## Tool 5: persist_assessment + trace_evidence (FastMCP)

In [ ]:
# Write a test assessment
result = execute_tool('persist_assessment', {
    'entity_id':     'LOAN-0002',
    'entity_type':   'LoanApplication',
    'regulation_id': 'APG-223',
    'verdict':       'REQUIRES_REVIEW',
    'confidence':    0.95,
    'findings': [{
        'finding_type': 'compliance_breach',
        'severity':     'HIGH',
        'description':  'LVR=95% breaches APG-223-THR-008 (LVR >= 90% requires senior review)',
        'pattern_name': 'high_lvr_loans',
    }],
    'reasoning_steps': [{
        'description':  'Retrieved LVR=95 from LoanApplication node via traverse_compliance_path',
        'cypher_used':  "MATCH (l:LoanApplication {loan_id: 'LOAN-0002'}) RETURN l.lvr",
        'section_ids':  ['APG-223-S4'],
        'chunk_ids':    [],
    }],
    'agent': 'notebook_test',
})
print('persist_assessment:', result)

assessment_id = result.get('assessment_id')

# Trace it back
if assessment_id:
    evidence = execute_tool('trace_evidence', {'assessment_id': assessment_id})
    print('\ntrace_evidence:')
    print(f'  Assessment: {evidence.get("assessment")}')
    print(f'  Findings:   {evidence.get("findings")}')
    print(f'  Steps:      {[s["description"] for s in evidence.get("reasoning_steps", [])]}')
    print(f'  Cited sections: {[s["section_id"] for s in evidence.get("cited_sections", [])]}')

## Summary

All 7 tools tested. Ready to proceed to `313_compliance_agent.ipynb`.